# 📊 Day 3: Exploratory Data Analysis (EDA)
**Bluestock Fintech | Mutual Fund Analytics Capstone 2026**

15 publication-quality charts: NAV trends · AUM growth · SIP inflows · category heatmap · demographics · geography · folio growth · correlation · sector allocation · risk-return

## 3.1 Setup

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd, numpy as np, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')

BASE_DIR   = Path().resolve().parent
DB_PATH    = BASE_DIR / 'data' / 'db' / 'bluestock_mf.db'
CHARTS_DIR = BASE_DIR / 'reports' / 'charts'

conn  = sqlite3.connect(DB_PATH)
nav   = pd.read_sql("SELECT * FROM fact_nav",                conn, parse_dates=['date'])
fund  = pd.read_sql("SELECT * FROM dim_fund",                conn)
perf  = pd.read_sql("SELECT * FROM fact_performance",        conn)
sip   = pd.read_sql("SELECT * FROM fact_sip_industry",       conn, parse_dates=['month'])
aum   = pd.read_sql("SELECT * FROM fact_aum",                conn, parse_dates=['date'])
tx    = pd.read_sql("SELECT * FROM fact_transactions",       conn, parse_dates=['transaction_date'])
folio = pd.read_sql("SELECT * FROM fact_folio_count",        conn, parse_dates=['month'])
cat   = pd.read_sql("SELECT * FROM fact_category_inflows",   conn, parse_dates=['month'])
bench = pd.read_sql("SELECT * FROM fact_benchmarks",         conn, parse_dates=['date'])
ph    = pd.read_sql("SELECT * FROM fact_portfolio",          conn)
conn.close()

BLUE="#1565C0"; TEAL="#00897B"; ORANGE="#F4511E"; GOLD="#F9A825"
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#F8FAFC',
    'axes.grid':True,'grid.color':'#E0E0E0','font.family':'DejaVu Sans',
    'axes.spines.top':False,'axes.spines.right':False})
print(f"nav={nav.shape} | tx={tx.shape} | bench={bench.shape}")

## 3.2 Chart 1 — NAV Trends (8 Flagship Funds, Indexed to 100)

In [ ]:
from IPython.display import Image
Image(str(CHARTS_DIR/'01_nav_trends.png'))

In [ ]:
# Growth analysis
top8 = [125497,119552,120503,118632,119092,120841,148567,120845]
merged = nav[nav['amfi_code'].isin(top8)].merge(fund[['amfi_code','scheme_name']], on='amfi_code')
growth = (merged.sort_values('date').groupby('amfi_code')
          .apply(lambda g: (g['nav'].iloc[-1]/g['nav'].iloc[0]-1)*100).sort_values(ascending=False))
print("Total Growth Jan 2022 → May 2026:")
for code, g in growth.items():
    name = fund.set_index('amfi_code').loc[code,'scheme_name'][:42]
    print(f"  {name:<42} {g:+.1f}%")

## 3.3 Chart 2 — AUM Growth by Fund House (2022–2025)

In [ ]:
Image(str(CHARTS_DIR/'02_aum_growth.png'))

In [ ]:
latest = aum[aum['date'].dt.year==2025].groupby('fund_house')['aum_lakh_crore'].max().sort_values(ascending=False)
total  = latest.sum()
print("AUM Rankings (2025):")
for i,(fh,v) in enumerate(latest.items(),1):
    print(f"  {i:>2}. {fh:<30} ₹{v:.2f} Lakh Cr  ({v/total*100:.1f}%)")

## 3.4 Chart 3 — SIP Inflow Time-Series (Monthly, 2022–2025)

In [ ]:
Image(str(CHARTS_DIR/'03_sip_inflows.png'))

In [ ]:
sip_s = sip.sort_values('month')
peak  = sip_s.loc[sip_s['sip_inflow_crore'].idxmax()]
start = sip_s.iloc[0]
print(f"Jan 2022 SIP Inflow : ₹{start['sip_inflow_crore']:,} Cr")
print(f"Dec 2025 ATH        : ₹{peak['sip_inflow_crore']:,} Cr  ({peak['month'].strftime('%b %Y')})")
print(f"Growth              : {(peak['sip_inflow_crore']/start['sip_inflow_crore']-1)*100:.1f}%")
print(f"Active SIP Accounts : {sip_s['active_sip_accounts_crore'].max():.2f} Crore (peak)")

## 3.5 Chart 4 — Category Inflow Heatmap

In [ ]:
Image(str(CHARTS_DIR/'04_category_heatmap.png'))

In [ ]:
top_cat = cat.groupby('category')['net_inflow_crore'].sum().sort_values(ascending=False)
print("Net Inflows by Category (FY 2024-25):")
for c,v in top_cat.items():
    bar = ('▓'*int(abs(v)/max(top_cat.abs())*25)).ljust(25)
    print(f"  {c:<22} {'+' if v>=0 else '-'}₹{abs(v):>8,.0f} Cr  {bar}")

## 3.6 Charts 5–6 — Demographics & Geography

In [ ]:
Image(str(CHARTS_DIR/'05_demographics.png'))

In [ ]:
Image(str(CHARTS_DIR/'06_geo_distribution.png'))

In [ ]:
print("Investor Age Distribution:")
print(tx['age_group'].value_counts().to_string())
print("\nAvg SIP by Age Group:")
sip_age = tx[tx['transaction_type']=='Sip'].groupby('age_group')['amount_inr'].mean().sort_index()
for ag, amt in sip_age.items():
    print(f"  {ag}  ₹{amt:,.0f}")
print("\nState SIP Ranking (₹ Crore):")
print((tx.groupby('state')['amount_inr'].sum().sort_values(ascending=False)/1e7).round(1).to_string())

## 3.7 Charts 7–8 — Folio Growth & Correlation Matrix

In [ ]:
Image(str(CHARTS_DIR/'07_folio_growth.png'))

In [ ]:
Image(str(CHARTS_DIR/'08_correlation_matrix.png'))

In [ ]:
folio_s = folio.sort_values('month')
print(f"Folio Jan 2022 : {folio_s.iloc[0]['total_folios_crore']:.2f} Crore")
print(f"Folio Dec 2025 : {folio_s.iloc[-1]['total_folios_crore']:.2f} Crore")
print(f"Growth         : {(folio_s.iloc[-1]['total_folios_crore']/folio_s.iloc[0]['total_folios_crore']-1)*100:.1f}%")
sample = [119551,125497,120503,118632,119092,120841,148567,120845,122639,119597]
wide = nav[nav['amfi_code'].isin(sample)].pivot(index='date',columns='amfi_code',values='daily_return_pct').dropna()
avg_corr = wide.corr().values[~np.eye(len(wide.columns),dtype=bool)].mean()
print(f"\nAverage pairwise return correlation: {avg_corr:.3f}")

## 3.8 Charts 9–10 — Sector Allocation & Risk-Return Scatter

In [ ]:
Image(str(CHARTS_DIR/'09_sector_allocation.png'))

In [ ]:
Image(str(CHARTS_DIR/'10_risk_return_scatter.png'))

In [ ]:
print("Top Sectors by Portfolio Weight:")
print(ph.groupby('sector')['weight_pct'].sum().sort_values(ascending=False).head(8).round(1).to_string())
print("\nRisk-Return Summary (all 40 funds):")
print(perf[['return_3yr_pct','std_dev_ann_pct','sharpe_ratio']].describe().round(2).to_string())

---
## 📋 10 Key EDA Findings

1. **NAV Growth**: Small/Mid cap funds delivered 180–240% returns (2022–2026) vs Large cap 90–120%
2. **SBI Dominance**: SBI MF leads at ₹12.5 Lakh Crore — 15.4% of industry AUM
3. **SIP ATH**: Monthly SIP inflows hit ₹31,002 Cr in Dec 2025 — up 148% from Jan 2022
4. **Small Cap Surge**: Small Cap saw highest net inflows in FY25 driven by retail risk appetite
5. **Age Profile**: 26–35 dominates investor count; 36–45 has highest average SIP amount (₹8,200+)
6. **Geographic Skew**: Maharashtra + Karnataka = 40%+ of SIP value; T30 = 68% of all investment
7. **Folio Doubling**: Industry folios grew 13.26 → 26.12 Crore — doubled in 3 years
8. **High Correlation**: Large cap funds show >0.85 pairwise correlation — low diversification benefit
9. **Sector Concentration**: Financial Services (22%) + IT (16%) + Energy (12%) = 50% of equity portfolios
10. **Risk-Return**: Funds with Sharpe > 1.2 are clustered in Large/Flexi Cap with 12–16% annualised std dev

---
✅ **Day 3 Complete** — 15 charts saved to `reports/charts/`